<a href="https://colab.research.google.com/github/Dkhan213/Etch-AI-Optimization/blob/main/src/etch_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install scikit-optimize

In [10]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================
# Quietly install the scikit-optimize machine learning library into Google Colab (-q means quiet)
!pip install scikit-optimize -q

import pandas as pd             # Handles data tables (like reading my CSV files)
import numpy as np              # Handles complex math and array calculations
from skopt import Optimizer     # The core AI engine that performs Bayesian Optimization
from skopt.space import Space, Real  # Allows me to define physical variable ranges (continuous numbers)
import warnings

# Hide harmless background warnings from the libraries to keep my output clean
warnings.filterwarnings('ignore')


# ==========================================
# GITHUB DATA SOURCES
# ==========================================
# Direct links to raw GitHub CSV files so Colab can read my data from anywhere
HISTORICAL_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/data/historical_baseline.csv"
NANOFAB_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/data/nanofab_experiments.csv"


# ==========================================
# STEP 1: DATA INGESTION & FEATURE ENGINEERING
# ==========================================
print("SYSTEM CHECK: Loading live data from GitHub...")

# Attempt to load the historical academic data to seed the AI's physical knowledge
try:
    external_df = pd.read_csv(HISTORICAL_SOURCE)
except Exception as e:
    # If GitHub is down or the link breaks, create an empty table so my script doesn't crash
    print(f"Notice: Could not load historical baseline ({e}). Using empty DataFrame.")
    external_df = pd.DataFrame(columns=['rf_power', 'chamber_pressure', 'sf6_flow', 'o2_flow', 'etch_rate'])

# Attempt to load our actual cleanroom experiments
try:
    nanofab_df = pd.read_csv(NANOFAB_SOURCE)
except Exception:
    nanofab_df = pd.DataFrame(columns=['rf_power', 'chamber_pressure', 'sf6_flow', 'o2_flow', 'etch_rate'])

# ACTIVE LEARNING & TRANSFER LEARNING SWITCH:
if len(nanofab_df) >= 3:
    print(f"Nanofab threshold reached ({len(nanofab_df)} runs logged). Training strictly on local UH hardware.")
    active_data = nanofab_df.copy()                           # Creates an explicit copy to prevent pandas slice warnings
    y_target = (-active_data['etch_rate']).values.tolist()    # Directly negates raw local etch rates for maximization
else:
    print(f"Insufficient Nanofab data ({len(nanofab_df)} runs logged). Seeding prior with {len(external_df)} records.")
    active_data = external_df.copy()                          # Creates an explicit copy of external baseline data

    # TARGET NORMALIZATION (MIN-MAX SCALING):
    # Scales literature etch rates between 0.0 and 1.0 so the AI learns the relative physical trends
    # (peaks and valleys) without being confused by high-pressure or high-power absolute numbers from other papers.
    if len(active_data) > 0:                                  # Checks that external data exists before attempting scaling
        min_rate = active_data['etch_rate'].min()             # Finds the lowest etch rate entry in the dataset
        max_rate = active_data['etch_rate'].max()             # Finds the highest etch rate entry in the dataset
        if max_rate != min_rate:                              # Prevents division by zero if all historical etch rates happen to be equal
            scaled_rate = (active_data['etch_rate'] - min_rate) / (max_rate - min_rate) # Applies standard Min-Max formula
        else:
            scaled_rate = active_data['etch_rate'] * 0 + 1.0   # Assigns uniform baseline score if max equals min
        y_target = (-scaled_rate).values.tolist()             # Negates normalized score so skopt can maximize it via minimization
    else:
        y_target = []                                         # Keeps target empty if no data is loaded

# FEATURE ENGINEERING:
# Computes O2/SF6 ratio so the AI learns the chemical passivation balance regardless of the total flow scale.
if len(active_data) > 0:
    active_data['o2_sf6_ratio'] = active_data['o2_flow'] / active_data['sf6_flow']

# Isolate feature matrix X (now includes 5 features due to the addition of chamber_pressure)
if len(active_data) > 0:
    X_prior = active_data[['rf_power', 'chamber_pressure', 'sf6_flow', 'o2_flow', 'o2_sf6_ratio']].values.tolist()
else:
    X_prior = []


# ==========================================
# STEP 2: DYNAMIC SEARCH SPACE & OPTIMIZER
# ==========================================
# DYNAMIC BOUND EXPANSION:
# Automatically stretches the AI's internal search space to fit high-flow or high-pressure literature papers
# while keeping our actual hardware output candidate selection strictly inside safe UH cleanroom bounds.
if len(X_prior) > 0:
    rf_min = min(20.0, float(active_data['rf_power'].min()))   # Dynamically finds minimum RF power in dataset
    rf_max = max(160.0, float(active_data['rf_power'].max()))  # Expands upper RF power bound if paper uses higher wattage
    p_min = min(5.0, float(active_data['chamber_pressure'].min()))    # Dynamically scales minimum pressure
    p_max = max(1000.0, float(active_data['chamber_pressure'].max())) # Dynamically scales max pressure (e.g., for 1 Torr papers)
    sf6_min = min(20.0, float(active_data['sf6_flow'].min()))  # Dynamically finds minimum SF6 flow
    sf6_max = max(50.0, float(active_data['sf6_flow'].max()))  # Expands upper SF6 bound for high-flow papers
    o2_min = min(5.0, float(active_data['o2_flow'].min()))     # Dynamically finds minimum O2 flow
    o2_max = max(50.0, float(active_data['o2_flow'].max()))    # Expands upper O2 bound for high-flow papers
    ratio_min = float(active_data['o2_sf6_ratio'].min())       # Finds lower O2/SF6 ratio bound
    ratio_max = float(active_data['o2_sf6_ratio'].max())       # Finds upper O2/SF6 ratio bound
else:
    # Fallback bounds if no data loads
    rf_min, rf_max = 20.0, 160.0
    p_min, p_max = 5.0, 1000.0
    sf6_min, sf6_max = 20.0, 200.0
    o2_min, o2_max = 5.0, 150.0
    ratio_min, ratio_max = 0.1, 1.0

search_space = [
    Real(rf_min, rf_max, name='rf_power'),
    Real(p_min, p_max, name='chamber_pressure'),               # Adds Chamber Pressure as an AI search dimension
    Real(sf6_min, sf6_max, name='sf6_flow'),
    Real(o2_min, o2_max, name='o2_flow'),
    Real(ratio_min, ratio_max, name='o2_sf6_ratio')            # Adds O2/SF6 ratio as a 5th continuous dimension
]

print("\nCALCULATING: Generating optimal parameters...")

# Configure the Bayesian AI Engine
ai_engine = Optimizer(
    dimensions=search_space,  # Tells the AI the 5 variable axes it is allowed to think about
    base_estimator="RF",      # Uses a "Random Forest" model to predict the etch rate landscape
    n_initial_points=0,       # Set to 0 to stop the AI from making random guesses; I want it to use our data immediately
    acq_func="EI",            # Expected Improvement: The math balancing exploring unknown areas vs exploiting known peaks
    random_state=42           # Locks the random seed so the script gives the exact same result if I run it twice
)

# STRICT UH CLEANROOM SAFETY LIMITS (Oxford System 100 Hardware Window)
# These bounds ensure the AI never suggests a recipe that will destroy our 1.2 micrometer S1813 mask
safe_limits = Space([
    Real(25.0, 60.0),   # Safe RF Substrate Power [W]: High enough to etch, low enough to save the photoresist
    Real(10.0, 30.0),   # Safe Chamber Pressure [mTorr]: Traps the AI near the optimal 25 mTorr peak
    Real(20.0, 45.0),   # Safe SF6 Gas Flow [SCCM]: Upper bound raised to 45.0 to allow 1:1 ratio
    Real(20.0, 45.0)    # Safe O2 Gas Flow [SCCM]: Upper bound raised to 45.0 to allow proper sidewall glass passivation
])

# Randomly generate 1,000 potential recipes that strictly obey the cleanroom safety limits above
safe_candidates_raw = safe_limits.rvs(1000, random_state=42)

# CANDIDATE FEATURE ALIGNMENT:
# Appends the calculated O2/SF6 ratio to each 4D safe recipe so it matches the 5D input format the AI expects.
safe_candidates_5d = []
for cand in safe_candidates_raw:
    ratio = cand[3] / cand[2]                                 # Computes candidate O2 flow divided by candidate SF6 flow
    safe_candidates_5d.append([cand[0], cand[1], cand[2], cand[3], ratio])

# TELL: Feed the isolated data (X and Y) into the AI so it learns the physics
if len(X_prior) > 0:
    ai_engine.tell(X_prior, y_target)                         # Feeds 5D feature matrix and normalized target array into skopt

    # SAFETY OVERRIDE:
    # Force the AI to evaluate all 1,000 safe 5D recipes and predict performance for each.
    predicted_scores = ai_engine.models[-1].predict(safe_candidates_5d)

    # Find the specific recipe index that yielded the best predicted performance score
    best_safe_idx = np.argmin(predicted_scores)

    # Lock in that specific safe recipe as our next experiment
    next_experiment = safe_candidates_raw[best_safe_idx]
else:
    # Fallback to the first safe candidate if both GitHub links fail entirely
    next_experiment = safe_candidates_raw[0]


# ==========================================
# STEP 3: OUTPUT PROTOCOL
# ==========================================
print("\n--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---")
print("1. Set static parameters on the Oxford System 100:")
print("   -> ICP Source Power: 800.0 [W]")
print("   -> Argon (Ar) Flow:  15.0  [SCCM]")
print("   -> Process Time:     60    [Seconds]")
print("\n2. Input the AI-OPTIMIZED variables:")
print(f"   -> RF Substrate Bias: {next_experiment[0]:.1f} [W]")
print(f"   -> Chamber Pressure:  {next_experiment[1]:.1f} [mTorr]")
print(f"   -> SF6 Gas Flow:      {next_experiment[2]:.1f} [SCCM]")
print(f"   -> O2 Gas Flow:       {next_experiment[3]:.1f} [SCCM]")

SYSTEM CHECK: Loading live data from GitHub...
Insufficient Nanofab data (less than 3 runs) (0 runs logged). Seeding prior with 27 records.

CALCULATING: Generating optimal parameters...

--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---
1. Set static parameters on the Oxford System 100:
   -> ICP Source Power: 800.0 [W]
   -> Argon (Ar) Flow:  15.0  [SCCM]
   -> Process Time:     60    [Seconds]

2. Input the AI-OPTIMIZED variables:
   -> RF Substrate Bias: 51.4 [W]
   -> Chamber Pressure:  25.2 [mTorr]
   -> SF6 Gas Flow:      42.7 [SCCM]
   -> O2 Gas Flow:       20.2 [SCCM]
